<!--nav--> [🗺 Learning path](README.md) · **34/46** · ◀ [The Hardware Roofline: NVIDIA vs AMD](./Hardware_Roofline_NVIDIA_vs_AMD.ipynb) · [Measuring GPU Code Honestly](./Measuring_GPU_Code_Honestly.ipynb) ▶

# GPU Architecture & CUDA Kernels: From a Memory Copy to Paged Attention

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sugeerth/gpu-training-notebooks/blob/main/GPU_Architecture_And_CUDA_Kernels.ipynb)

Every notebook in this repo so far has treated the GPU as a machine with two numbers —
bandwidth and FLOP/s — and reasoned about serving from there. That model is right, and it is
where [The Hardware Roofline](./Hardware_Roofline_NVIDIA_vs_AMD.ipynb) leaves you. But it
explains why a kernel is slow only in the way a speed limit explains a traffic jam.

This notebook opens the machine. It is built around **six CUDA programs that live in this
repository as real, compilable source** — [`kernels/`](https://github.com/sugeerth/gpu-training-notebooks/tree/main/kernels)
— not as code fragments quoted in a markdown cell. You can clone them, build them with `nvcc`,
read them in an editor, and break them. Each one implements the same function several ways and
reports what each way costs, as a fraction of what your card can actually do.

| | | |
|---|---|---|
| `01_copy.cu` | copying an array | coalescing — why the same bytes cost 10x more depending on who reads them |
| `02_reduce.cu` | summing an array | warps, divergence, shared-memory bank conflicts, shuffles |
| `03_sgemm.cu` | `C = A·B` | arithmetic intensity, tiling, register blocking |
| `04_rmsnorm.cu` | the first real LLM op | why fusion beats optimization when you are memory-bound |
| `05_dequant_gemv.cu` | decode-time matvec | why int4 makes decoding 4x faster and prefill barely faster at all |
| `06_flash_decode.cu` | decode attention | online softmax, FlashDecoding, PagedAttention |

**You do not need a GPU to run this notebook.** Every kernel also compiles with `g++`, against
a shim that runs one OS thread per CUDA thread with real barriers and real warp shuffles. On a
CPU you get correctness, not timings — and the notebook says so at every step rather than
reporting numbers that could be mistaken for GPU results.

**With a GPU (a free Colab T4 is plenty), you get the measurements**, taken with CUDA events,
after warmup, with an L2 flush between repetitions, reported as median and MAD, and divided by
your own card's peak. [Measuring GPU Code Honestly](./Measuring_GPU_Code_Honestly.ipynb) is
about why each of those clauses is there.

In [ ]:
# Setup. On Colab this clones the repo; run it and everything below just works.
import os, subprocess, sys, shutil, textwrap
from pathlib import Path

def find_repo():
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / "kernels" / "Makefile").exists():
            return cand
    return None

REPO = find_repo()
if REPO is None:
    dest = Path("/content/gpu-training-notebooks")
    if not (dest / "kernels" / "Makefile").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/sugeerth/gpu-training-notebooks", str(dest)],
                       check=True)
    REPO = dest
KERNELS = REPO / "kernels"
print("repo    :", REPO)

def sh(cmd, cwd=KERNELS, limit=6000):
    """Run a shell command and show its output. Returns the exit code."""
    p = subprocess.run(cmd, shell=True, cwd=str(cwd), capture_output=True, text=True)
    out = (p.stdout or "")[-limit:]
    if out:
        print(out, end="" if out.endswith("\n") else "\n")
    if p.returncode != 0 and p.stderr:
        print((p.stderr or "")[-2500:], file=sys.stderr)
    return p.returncode

def peek(filename, symbol, before=14):
    """Print one function from a kernel source file, with the comment block above it.

    The notebook reads the real file rather than carrying its own copy, so the code you see
    here is the code that CI compiles and that you would edit."""
    lines = (KERNELS / filename).read_text().split("\n")
    start = next((i for i, l in enumerate(lines) if symbol in l and ("(" in l or "=" in l)), None)
    if start is None:
        print(f"{symbol} not found in {filename}"); return
    # walk back over the leading comment block
    top = start
    while top > 0 and (lines[top - 1].startswith("//") or lines[top - 1].strip() == ""
                       or lines[top - 1].startswith("__device__")
                       or lines[top - 1].startswith("__global__")
                       or lines[top - 1].startswith("constexpr")):
        top -= 1
        if start - top > before * 6:
            break
    # walk forward to the closing brace at column 0
    end = start
    while end < len(lines) - 1 and lines[end] != "}":
        end += 1
    print("\n".join(lines[top:end + 1]))

# --- what are we running on? -------------------------------------------------------------
HAVE_NVCC = shutil.which("nvcc") is not None
try:
    import torch
    HAVE_GPU = torch.cuda.is_available()
    GPU_NAME = torch.cuda.get_device_name(0) if HAVE_GPU else None
except ImportError:
    torch, HAVE_GPU, GPU_NAME = None, False, None

if HAVE_NVCC and HAVE_GPU:
    MODE = f"GPU ({GPU_NAME}) — kernels compile with nvcc and report real timings"
elif HAVE_GPU:
    MODE = "GPU present but no nvcc — kernels run via the CPU shim (correctness only)"
else:
    MODE = "no GPU — kernels compile with g++ via cuda_shim.hpp (correctness only, no timings)"
print("mode    :", MODE)
print("nvcc    :", shutil.which("nvcc") or "not found")
print("g++     :", shutil.which("g++") or "not found")

## Part 1 · The machine you are writing for

A GPU is not a fast CPU. It is a machine that hides latency with parallelism instead of with
caches, and almost every design rule below follows from that one choice.

**Streaming Multiprocessors.** The card is a few dozen to a couple of hundred independent
processors (NVIDIA: SM; AMD: CU). An A100 has 108, an H100 has 132, a T4 has 40. A kernel
launch creates a *grid* of thread *blocks*; the hardware assigns each block to one SM and it
stays there. If your grid has fewer blocks than the GPU has SMs, the rest of the machine is
idle no matter how good the kernel is — which is exactly the problem `06_flash_decode.cu`
solves with split-K.

**Warps.** Inside a block, threads are grouped into warps of 32 (AMD: wavefronts of 64) that
share a single instruction pointer. This is the single most important fact about GPU
programming:

> 32 threads execute the same instruction, or they take turns.

A branch that splits a warp does not run at half cost. It runs *both sides*, with the inactive
lanes masked off — full cost, half the work. And a memory instruction issued by a warp is not
32 loads, it is one request that the memory system tries to satisfy with as few transactions as
it can. Both of those are measured in the kernels below.

**Latency hiding.** An HBM access costs 400–800 cycles. A CPU spends transistors on caches and
out-of-order execution to avoid paying that. A GPU just keeps more warps resident: when one
stalls on memory, the SM switches to another in a single cycle. An SM can hold up to 64 warps
(2048 threads) at once, and how many it *actually* holds — the **occupancy** — is decided by
how many registers and how much shared memory each block asks for. That is the budget Part 5
makes concrete.

In [ ]:
# What is actually in front of you, and what it implies.
REFERENCE = {
    # name        SMs   HBM GB/s  fp32 TF  fp16 TC TF  L2 MB  shared KB/SM
    "T4":        (40,    320,      8.1,     65,        4,     64),
    "V100":      (80,    900,     15.7,    125,        6,     96),
    "A100":     (108,   2039,     19.5,    312,       40,    164),
    "L4":        (58,    300,     30.3,    121,       48,    100),
    "H100":     (132,   3350,     67.0,    990,       50,    228),
}

def describe(sms, gbps, tf32, tf16, l2mb, shkb, name):
    ridge32 = tf32 * 1e12 / (gbps * 1e9)
    ridge16 = tf16 * 1e12 / (gbps * 1e9)
    print(f"{name:<10} {sms:>4} SMs {gbps:>6.0f} GB/s {tf32:>6.1f} TF32 {tf16:>6.0f} TF16"
          f" {l2mb:>4.0f} MB L2 {shkb:>4.0f} KB shm/SM   ridge {ridge32:>5.1f} / {ridge16:>5.0f} FLOP/byte")

print("reference cards (ridge point = FLOP per byte above which a kernel is compute-bound):\n")
for name, spec in REFERENCE.items():
    describe(*spec, name)

if HAVE_GPU:
    p = torch.cuda.get_device_properties(0)
    print(f"\nyour card: {p.name}")
    print(f"  {p.multi_processor_count} SMs x up to {p.max_threads_per_multi_processor} threads"
          f" = {p.multi_processor_count * p.max_threads_per_multi_processor:,} threads resident at once")
    print(f"  {getattr(p, 'L2_cache_size', 0)/1048576:.0f} MB L2, "
          f"{p.shared_memory_per_block/1024:.0f} KB shared memory per block, "
          f"{p.total_memory/1e9:.0f} GB HBM")
    print(f"  warp size {p.warp_size}, compute capability {p.major}.{p.minor}")
else:
    print("\n(no GPU here — the table above is what you would be reading off. Every kernel")
    print(" below still runs; it just reports correctness instead of time.)")

### The memory hierarchy, and the only number that matters about it

| where | size | latency | bandwidth | who shares it |
|---|---|---|---|---|
| registers | 256 KB per SM | ~1 cycle | ~20 TB/s | one thread |
| shared memory / L1 | 100–228 KB per SM | ~30 cycles | ~10 TB/s | one block |
| L2 | 4–50 MB | ~200 cycles | ~5 TB/s | the whole GPU |
| HBM | 16–192 GB | 400–800 cycles | 0.3–3 TB/s | the whole GPU |

Each step down is roughly an order of magnitude worse and an order of magnitude bigger. Every
optimization in this notebook is one of exactly two moves:

1. **move data up the hierarchy and reuse it there** (tiling, in `03_sgemm.cu`)
2. **need fewer bytes from the bottom of it** (fusion in `04_rmsnorm.cu`, quantization in
   `05_dequant_gemv.cu`)

There is no third move. When you read a kernel optimization writeup, it is one of these two,
and it is worth deciding which before reading the code.

In [ ]:
# Coalescing, quantified — the rule 01_copy.cu is built around.
#
# When a warp issues one load instruction, the memory system covers the 32 addresses with
# 32-byte sectors. Consecutive floats need 4 sectors; scattered ones need up to 32.
import uuid, json
from IPython.display import HTML, display

def sectors_per_warp(stride_elems, elem_bytes=4, sector=32):
    """How many 32-byte sectors one warp's load touches, at a given element stride."""
    addrs = [i * stride_elems * elem_bytes for i in range(32)]
    return len({a // sector for a in addrs})

rows = []
for stride in (1, 2, 4, 8, 16, 32, 64):
    s = sectors_per_warp(stride)
    rows.append({"stride": stride, "sectors": s,
                 "useful": round(100 * (32 * 4) / (s * 32), 1)})

print(f"{'element stride':>15} {'sectors moved':>15} {'bytes moved':>13} {'% you asked for':>17}")
for r in rows:
    print(f"{r['stride']:>15} {r['sectors']:>15} {r['sectors']*32:>10} B {r['useful']:>16.1f}%")
print("\nEvery row wants the same 128 bytes. The last one moves 1024 to get them.")

D3_URL = "https://cdn.jsdelivr.net/npm/d3@7/dist/d3.min.js"
def show_d3(js, data=None, height=300):
    div = f"viz_{uuid.uuid4().hex[:10]}"
    html = f"""
<div id="{div}" style="width:100%;max-width:760px;font-family:system-ui,sans-serif"></div>
<script>
(function() {{
  function go() {{
    const DATA = {json.dumps(data)};
    const root = d3.select("#{div}");
    {js}
  }}
  if (window.d3) go();
  else {{ const s = document.createElement("script"); s.src = "{D3_URL}";
          s.onload = go; document.head.appendChild(s); }}
}})();
</script>"""
    display(HTML(html))

show_d3(r"""
  const W = 700, H = 260, M = {t: 26, r: 20, b: 40, l: 110};
  const svg = root.append("svg").attr("width", W).attr("height", H);
  const x = d3.scaleLinear().domain([0, 32]).range([M.l, W - M.r]);
  const y = d3.scaleBand().domain(DATA.map(d => "stride " + d.stride))
              .range([M.t, H - M.b]).padding(0.25);
  svg.append("g").attr("transform", `translate(0,${H - M.b})`)
     .call(d3.axisBottom(x).ticks(8)).selectAll("text").style("font-size", "11px");
  svg.append("g").attr("transform", `translate(${M.l},0)`).call(d3.axisLeft(y))
     .selectAll("text").style("font-size", "12px").style("font-family", "ui-monospace, monospace");
  svg.selectAll("rect.bar").data(DATA).join("rect").attr("class", "bar")
     .attr("x", x(0)).attr("y", d => y("stride " + d.stride)).attr("height", y.bandwidth())
     .attr("width", d => x(d.sectors) - x(0))
     .attr("fill", d => d.sectors === 4 ? "#2a9d5c" : d3.interpolateOrRd(0.25 + d.sectors / 48));
  svg.selectAll("text.lab").data(DATA).join("text").attr("class", "lab")
     .attr("x", d => x(d.sectors) + 6).attr("y", d => y("stride " + d.stride) + y.bandwidth() / 2 + 4)
     .style("font-size", "11px").style("font-family", "ui-monospace, monospace")
     .text(d => d.sectors + " sectors — " + d.useful + "% useful");
  svg.append("text").attr("x", M.l).attr("y", 16).style("font-size", "12px")
     .text("32-byte sectors moved by ONE warp executing ONE load instruction");
""", rows, height=280)

## Part 2 · `01_copy.cu` — coalescing

Copying an array has no arithmetic at all, which makes it the cleanest possible place to see
the rule above. The trap the file is built around is that the *intuitive* decomposition is the
bad one:

```
thread 0 takes elements [0, C)        <- how you would split work across CPU cores,
thread 1 takes elements [C, 2C)          where each core wants its own cache lines
...
```

On a CPU that is right — private cache lines per core. On a GPU the 32 threads of a warp share
an instruction, so they must share a cache line too, and this layout puts every lane in a
different sector. The fix is the **grid-stride loop**: lane `t` handles element `base + t`, so
the warp's 32 addresses are 32 consecutive floats.

In [ ]:
peek("01_copy.cu", "copy_thread_chunks")
print("\n" + "-" * 88 + "\n")
peek("01_copy.cu", "copy_coalesced")

In [ ]:
# Build and run it. On a GPU: four timings against your card's peak bandwidth.
# On a CPU: four correctness checks and no timings.
sh("make --no-print-directory 01_copy")

Two further variants in that file are worth knowing about, because they are the difference
between "coalesced" and "fast":

- **`copy_vec4`** loads 16 bytes per thread instead of 4. The bytes moved are identical, but
  the instruction count drops 4x and so does the number of in-flight requests needed to cover
  the memory latency. Scalar copy typically reaches 70–80% of achievable bandwidth; `float4`
  reaches 90%+.
- **`copy_vec4_unroll4`** issues four independent loads before consuming any of them. This is
  Little's Law applied to a memory system: `bytes in flight = bandwidth × latency`. An H100 at
  3 TB/s with 500 ns of latency needs ~1.5 MB *outstanding* at every instant. You get there
  either with ~100k concurrent threads, or with fewer threads each holding several independent
  loads.

## Part 3 · `02_reduce.cu` — warps, divergence, and bank conflicts

Summing an array is the second-simplest kernel and, unlike a copy, it is *not* embarrassingly
parallel: every thread's partial result has to reach every other thread's. That makes it the
standard vehicle for the three mechanisms that decide the speed of anything cooperative — and
softmax, layernorm, RMSNorm and attention are all cooperative.

**Shared memory is 32 banks, 4 bytes wide, striped.** Address `a` lives in bank `(a/4) % 32`.
One bank serves one address per cycle. If the 32 lanes of a warp hit 32 different banks, the
access takes one cycle; if they all hit the same bank at different addresses, it takes 32,
serialized. This is invisible in the source — it is a property of the *index arithmetic*.

In [ ]:
# Bank conflicts, computed from the rule rather than asserted.
def conflict_degree(index_of_lane, banks=32):
    """Worst-case serialization factor for one warp's shared-memory access."""
    hits = {}
    for lane in range(32):
        b = (index_of_lane(lane) * 4 // 4) % banks   # float-indexed shared array
        hits.setdefault(b, set()).add(index_of_lane(lane))
    return max(len(v) for v in hits.values())

print("the same reduction tree, written two ways, at each step:\n")
print(f"{'step':>6} {'stride':>7} {'interleaved: s[2*stride*tid]':>32} {'sequential: s[tid]':>22}")
for step in range(5):
    stride = 2 ** step
    inter = conflict_degree(lambda lane, s=stride: 2 * s * lane)
    seq = conflict_degree(lambda lane: lane)
    print(f"{step:>6} {stride:>7} {inter:>29}-way {seq:>19}-way")
print("\nInterleaved addressing degrades every step until one bank serves the whole warp.")
print("Sequential addressing — s[tid] += s[tid + stride] — is conflict-free at every step,")
print("and it is the *only* difference between variants 2 and 3 in the file.")

In [ ]:
peek("02_reduce.cu", "warp_reduce_sum")

`__shfl_down_sync` is the third mechanism, and the reason modern reductions barely touch shared
memory: a lane reads another lane's **register** directly, in one instruction, with no memory
and no barrier. A 256-thread block reduction goes from 8 barriers and 8 shared-memory round
trips to 1 barrier and 8 floats of shared traffic.

The mask argument is not decoration. `0xffffffff` asserts that all 32 lanes are present at this
instruction. Since Volta, lanes can be at different instructions, and a shuffle whose mask names
a lane that is not there is undefined behaviour — a bug that appears only under divergence, only
on some inputs.

In [ ]:
sh("make --no-print-directory 02_reduce")

## Part 4 · `03_sgemm.cu` — arithmetic intensity and tiling

A copy has intensity 0 FLOP/byte. A matrix multiply of size `n` has `2n³` FLOPs over `3n²`
floats, so its intensity is `O(n)` — it can be as compute-bound as you like, **if you get the
reuse**. Getting the reuse is the entire job.

The file has four variants, and the first pair is the famous one: `sgemm_naive_uncoalesced`
and `sgemm_naive_coalesced` differ only in whether `threadIdx.x` indexes the row or the column
of `C`. Same FLOPs, same algorithm, same occupancy, typically 5–10x apart. It is worth looking
at how invisible that is in the source.

In [ ]:
# The two levels of tiling fix two different bottlenecks. Worth keeping them apart.
#
#   HBM intensity is set by the BLOCK tile.  A block computing BM x BN of C reads
#   (BM + BN) x K values to do 2*BM*BN*K FLOPs  ->  BM*BN / (2*(BM+BN))  FLOP/byte.
#
#   Shared-memory traffic is set by the THREAD tile. Each thread reads TM + TN floats from
#   shared to do TM * TN fused multiply-adds.
def hbm_intensity(bm, bn):
    return 2 * bm * bn / (4 * (bm + bn))      # FLOP per byte fetched from HBM

def shared_fmas_per_read(tm, tn):
    return tm * tn / (tm + tn)                 # FLOPs per float read from shared memory

VARIANTS = [("naive (no tile)", 1, 1, 1, 1),
            ("shared tiling, TILE=16", 16, 16, 1, 1),
            ("+ register block 4x4", 64, 64, 4, 4),
            ("cuBLAS-class, 128 / 8x8", 128, 128, 8, 8)]

print("HBM arithmetic intensity, and whether it clears each card's fp32 ridge:\n")
hdr = f"{'variant':<26}{'FLOP/byte':>11}{'shared FMA/read':>17}   "
print(hdr + "".join(f"{c:>9}" for c in REFERENCE))
for name, bm, bn, tm, tn in VARIANTS:
    ai = hbm_intensity(bm, bn)
    row = f"{name:<26}{ai:>11.2f}{shared_fmas_per_read(tm, tn):>17.2f}   "
    for card, spec in REFERENCE.items():
        r32 = spec[2] * 1e12 / (spec[1] * 1e9)
        row += f"{'compute' if ai > r32 else 'memory':>9}"
    print(row)

print("\nfp32 ridge point per card: " + ", ".join(
    f"{c} {spec[2]*1e12/(spec[1]*1e9):.0f}" for c, spec in REFERENCE.items()))
print("""
Three things fall out of that table:

  * Register blocking does NOT raise HBM intensity. Variant 4 in 03_sgemm.cu wins the HBM
    argument purely by having a 64-wide block tile instead of 16; what the 4x4 thread tile
    buys is 4x less *shared memory* traffic per FMA, which is the next constraint along.
  * A square block tile of width T gives T/4 FLOP/byte, so clearing an A100's fp32 ridge of
    ~10 needs T >= 39. That is why real GEMMs use 128-wide tiles and not 16.
  * An L4 has ~30 TFLOP/s fp32 against 300 GB/s — a ridge of ~101. No block tile that fits in
    shared memory can cross it. On parts like that, fp32 SIMT GEMM cannot be made
    compute-bound at all, and tensor cores are not an optimization but the only route to the
    machine's arithmetic. That is Part 9.""")

In [ ]:
peek("03_sgemm.cu", "sgemm_tiled")

The two `__syncthreads()` in that loop are both mandatory, and they are the two places this
kernel is most often broken:

- after the stores, so nobody reads a tile before it is written
- after the inner loop, so nobody overwrites a tile another thread is still reading

Drop the second one and the kernel still passes on small inputs, because with one warp per
block there is nobody to race. It fails at scale, intermittently. The CPU shim in this repo
runs real threads with real barriers, so it fails there too — deterministically, in
milliseconds. `tools/verify_kernels.py` injects exactly that bug on every CI run and requires
that it be caught.

In [ ]:
sh("make --no-print-directory 03_sgemm")

## Part 5 · Occupancy — the budget that decides everything else

An SM can hold up to 2048 threads, but only if each of them is cheap. Three resources are
divided among resident blocks, and whichever runs out first sets the occupancy:

- **registers**: 65536 per SM (256 KB). At 64 registers per thread you fit 1024 threads.
- **shared memory**: 100–228 KB per SM, divided among resident blocks.
- **block slots**: at most 16–32 blocks per SM regardless of how small they are.

Low occupancy is not automatically bad — a kernel with enough instruction-level parallelism can
saturate memory with few warps, which is what `copy_vec4_unroll4` exploits. But for a
latency-bound kernel with a dependent chain, occupancy *is* the latency-hiding budget.

In [ ]:
def occupancy(threads_per_block, regs_per_thread, shmem_per_block_kb,
              regs_per_sm=65536, shmem_per_sm_kb=164, max_threads_sm=2048, max_blocks_sm=32):
    """Which resource runs out first, and how much of the SM you end up using."""
    by_regs = regs_per_sm // max(1, regs_per_thread * threads_per_block)
    by_shmem = int(shmem_per_sm_kb // shmem_per_block_kb) if shmem_per_block_kb else max_blocks_sm
    by_threads = max_threads_sm // threads_per_block
    blocks = min(by_regs, by_shmem, by_threads, max_blocks_sm)
    limiter = min([(by_regs, "registers"), (by_shmem, "shared memory"),
                   (by_threads, "thread slots"), (max_blocks_sm, "block slots")])[1]
    return blocks, blocks * threads_per_block / max_threads_sm, limiter

print(f"{'block':>6} {'regs/thr':>9} {'shmem KB':>9} {'blocks/SM':>10} {'occupancy':>10}  limited by")
for cfg in [(256, 32, 0), (256, 64, 0), (256, 128, 0), (256, 32, 8), (256, 32, 48),
            (1024, 64, 0), (128, 40, 4), (64, 32, 0)]:
    b, occ, lim = occupancy(*cfg)
    print(f"{cfg[0]:>6} {cfg[1]:>9} {cfg[2]:>9} {b:>10} {occ:>9.0%}  {lim}")

print("\nRead the last row carefully: a 64-thread block hits the block-slot cap long before")
print("it hits any resource limit, so half the SM sits empty however cheap the threads are.")
print("That is why 128-256 threads per block is the near-universal default.")

## Part 6 · `04_rmsnorm.cu` — the first real LLM op, and why fusion wins

$$y_i = \frac{x_i}{\sqrt{\text{mean}(x^2) + \epsilon}} \cdot w_i$$

Two FLOPs of useful arithmetic per element against 8 bytes of traffic: 0.25 FLOP/byte, against
a ridge point of ~10–100. RMSNorm will never be compute-bound on any GPU that will ever be
built. **There is nothing to make faster inside this kernel.**

So the only lever is the byte count — and that is not a property of the kernel, it is a
property of how many kernels there are. A transformer block does

```
h  = x + attn_out         # elementwise add:  reads 2, writes 1
hn = rmsnorm(h) * w       # reads 1, writes 1
```

as two launches, and the intermediate makes a full round trip to HBM even though the next
kernel reads it back immediately. Fusing them does not make the arithmetic faster; it deletes
the round trip. Counted in trips over the hidden state: 5 unfused, 4 fused — a 20% cut, applied
to two normalizations in every one of 80 layers, on every decode step.

In [ ]:
sh("make --no-print-directory 04_rmsnorm")

Watch the `GB/s` column when you run this on a GPU: **every variant runs at close to the same
bandwidth**. They differ in how many bytes they need, not in how fast they move them. That
distinction is the whole subject of memory-bound optimization, and it is why profiling a fused
kernel and finding it "only" at 85% of peak bandwidth means you are finished.

## Part 7 · `05_dequant_gemv.cu` — why quantization is a decode technique

During decode there is one new token per sequence, so every weight matrix is applied to a
*vector*. For `y = Wx` with `W` of shape `[N, K]`:

- arithmetic: `2NK` FLOPs
- traffic: `NK × bytes-per-weight` — `x` and `y` are negligible, `W` is everything
- intensity: `2 / bytes_per_weight` FLOP/byte

which is **0.5 for fp32, 1.0 for fp16, 2.0 for int8, 4.0 for int4** — every one of them one to
two orders of magnitude below the ridge point. A GEMV is memory-bound no matter what, and that
has a blunt consequence:

> Decode time is proportional to the number of **bytes** in the weights. Nothing else about the
> arithmetic matters.

int4 weights decode ~4x faster than fp16 — not because int4 arithmetic is fast (the kernel
converts every weight back to fp32 and does fp32 math), but because there are a quarter as many
bytes to fetch. The dequantization is free: it happens in registers, on data already paid for,
on a machine with nothing else to do.

The same quantization does almost nothing for **prefill**, where `W` is applied to hundreds of
tokens at once, intensity is in the hundreds, and the kernel is compute-bound. One technique,
two entirely different reasons, and they do not both apply at once — which is the argument
[Quantized Serving Showdown](./Quantized_Serving_Showdown.ipynb) measures end to end.

In [ ]:
sh("make --no-print-directory 05_dequant_gemv")

In [ ]:
peek("05_dequant_gemv.cu", "gemv_int4")

## Part 8 · `06_flash_decode.cu` — online softmax, split-K, paged KV

This is the file the rest of the repo has been building toward.
[Attention Kernels From Scratch](./Attention_Kernels_From_Scratch.ipynb) develops these four
algorithms in NumPy — that notebook is the specification, and this file is the machine.

Everything rests on one identity. For two partial results $(m_1,\ell_1,\text{acc}_1)$ and
$(m_2,\ell_2,\text{acc}_2)$ with $m = \max(m_1,m_2)$:

$$\ell = \ell_1 e^{m_1-m} + \ell_2 e^{m_2-m} \qquad \text{acc} = \text{acc}_1 e^{m_1-m} + \text{acc}_2 e^{m_2-m}$$

It is associative and commutative, which is exactly why the KV cache can be split any way the
scheduler likes and merged in any order.

**One honest caveat, because this is usually oversold.** In *prefill*, online softmax eliminates
the `N × N` score matrix, and that is worth an order of magnitude — the FlashAttention result.
In *decode*, the intermediate is a vector of `S` scores against a KV cache of `S × D`, so the
traffic saved is `3/(2D)` ≈ **1.2% at D=128**. Nearly nothing. What the two-pass version
actually costs at decode time is `S` floats of scratch per head per layer that must be
*reserved* for the longest admissible sequence — 16 MB per layer at 128k context and 32 heads.

The large win in that file is split-K, and it is an **occupancy** win, not a traffic win: one
query means one block per head means 32 blocks on a 132-SM GPU.

In [ ]:
sh("make --no-print-directory 06_flash_decode")

In [ ]:
# The whole of PagedAttention, in the inner loop.
peek("06_flash_decode.cu", "decode_paged")

The entire difference between the contiguous kernel and the paged one is:

```c
const int page = table[h * npages + j0 / TILE];   // <- one extra load, once per page
```

off a table small enough to be L2-resident forever. In exchange, the allocator never has to
find `S` contiguous tokens' worth of memory, so there is no external fragmentation and no
over-reservation for a sequence's *maximum* length. That trade is the reason vLLM exists, and
the cost of it is visible right there.

## Part 9 · Tensor cores — the ceiling these kernels do not reach

Everything above uses the fp32 SIMT pipeline, which is the right way to learn but is not where
a modern GPU's arithmetic lives. Since Volta, each SM also has **tensor cores**: units that
execute a small matrix multiply-accumulate as a single instruction, typically `16×16×16` per
warp.

The gap is not a detail. On an A100 that is 19.5 TFLOP/s fp32 against 312 TFLOP/s fp16 tensor —
**16x**. On an H100, 67 against 990. Which means:

- the ridge point moves by the same factor: A100 fp32 ridge ≈ 10 FLOP/byte, fp16 tensor ≈ 153.
  Kernels that were comfortably compute-bound become memory-bound the moment you use tensor
  cores, which is why *fusion* and *quantization* got more important, not less, as arithmetic
  got cheaper.
- `03_sgemm.cu`'s register-blocked variant, at maybe 60–80% of fp32 peak, is still ~15x slower
  than cuBLAS on the same card. It is not a bad kernel. It is a kernel using the wrong units.

Writing tensor-core kernels by hand means `wmma` or inline `mma.sync` PTX, plus a swizzled
shared-memory layout to avoid bank conflicts on the fragment loads. That is genuinely a
different discipline, and the honest advice is: **use CUTLASS, cuBLAS, or Triton**. The value of
having written `03_sgemm.cu` is that you now know what those libraries are doing and why their
tile sizes are what they are.

`kernels/` deliberately contains no hand-written tensor-core kernel, for a reason worth being
explicit about: every kernel in that directory is verified by CI on a machine with no GPU, and
a `wmma` kernel cannot be. Shipping one unverified alongside six verified ones would quietly
undermine the guarantee the directory makes. The cell below measures the gap on your own card
instead.

In [ ]:
# The tensor-core gap, measured rather than quoted.
if HAVE_GPU:
    import time as _t
    def timed(fn, reps=20, warmup=5):
        for _ in range(warmup): fn()
        torch.cuda.synchronize()
        s, e = torch.cuda.Event(True), torch.cuda.Event(True)
        ts = []
        for _ in range(reps):
            s.record(); fn(); e.record(); e.synchronize()
            ts.append(s.elapsed_time(e))
        ts.sort()
        return ts[len(ts)//2]

    n = 4096
    flops = 2.0 * n ** 3
    for dtype, label in ((torch.float32, "fp32 (SIMT)"),
                         (torch.float16, "fp16 (tensor cores)"),
                         (torch.bfloat16, "bf16 (tensor cores)")):
        try:
            a = torch.randn(n, n, dtype=dtype, device="cuda")
            b = torch.randn(n, n, dtype=dtype, device="cuda")
            ms = timed(lambda: torch.mm(a, b))
            print(f"{label:<22} {ms:7.3f} ms   {flops/(ms/1e3)/1e12:7.1f} TFLOP/s")
            del a, b
        except RuntimeError as exc:
            print(f"{label:<22} unavailable: {exc}")
    torch.cuda.empty_cache()
    print("\nThe ratio between the first row and the others is the factor 03_sgemm.cu")
    print("leaves on the table by using the fp32 pipeline.")
else:
    print("No GPU here. On an A100 this prints roughly:")
    print("  fp32 (SIMT)             11.5 ms      12.0 TFLOP/s")
    print("  fp16 (tensor cores)      0.9 ms     150.0 TFLOP/s")
    print("  bf16 (tensor cores)      0.9 ms     150.0 TFLOP/s")
    print("\n...a ~13x gap, against a 16x theoretical one.")

## What to take away

1. **A warp is the unit of everything.** 32 threads, one instruction. Divergence costs both
   sides; a scattered load costs up to 32 transactions for 128 bytes of data.
2. **There are only two optimizations**: move data up the memory hierarchy and reuse it there,
   or need fewer bytes from the bottom of it. Every technique in this repo is one of the two.
3. **Most LLM inference kernels are memory-bound**, so the second one dominates. That is why
   fusion and quantization are the workhorses and why "more FLOPs are free" is literally true
   at decode time.
4. **Occupancy is a budget, not a score.** Registers, shared memory and block slots; whichever
   runs out first sets it, and a kernel with enough independent loads in flight does not need
   much of it.
5. **A number without a ceiling is not a measurement.** Every table in `kernels/` divides by
   what the card can actually do, which is the only way to tell a good kernel from a fast card.

### Next

- [Measuring GPU Code Honestly](./Measuring_GPU_Code_Honestly.ipynb) — the other half of this
  notebook: why a benchmark number is usually wrong, and the six mechanical reasons.
- [Attention Kernels From Scratch](./Attention_Kernels_From_Scratch.ipynb) — the algorithms in
  `06_flash_decode.cu`, developed in NumPy with an animated tile sweep.
- [Portable Kernels & Precision](./Portable_Kernels_Precision_Matrix.ipynb) — the same kernels
  in Triton, and what actually runs on AMD.
- [Anatomy of a Decode Step](./Anatomy_Of_A_Decode_Step.ipynb) — zoom back out: where these
  microseconds land in a real token.

### Try breaking something

The fastest way to trust any of this is to make it fail. In `kernels/`:

```bash
# delete a __syncthreads() from sgemm_blocked and watch the shim catch the race
make check
# then let CI's fault injector prove the checks can fail at all
python ../tools/verify_kernels.py
```